# 疾患関連遺伝子のスコアリング

候補リストから毎回ランダムに一群を取り出し、アルファベットのラベルを振って出題し、
遺伝子ごとのスコアを繰り返し平均する。

| | |
|---|---|
| ① | 疾患名を入力 |
| ② | ローカルで動いているモデルから選択 |
| ③ | 遺伝子リストを読み込む |
| ④ | プロンプト |
| ⑤ | `ranker()` — 1 群を採点して {遺伝子: スコア} を返す |
| ⑥ | ランダムに選んで繰り返し、平均する |
| ⑦ | DataFrame で表示 |

---

### 先に断っておく制約

**1. 1 群あたりの遺伝子は最大 26 個。** アルファベットは 26 文字しかないので、
27 個すべてに 1 文字を割り当てることはできません。2 文字（AA, AB…）にすると複数トークンになり、
次の 1 トークンだけを見る採点方式が壊れます。既定は 26 個（A〜Z）。
逃げ道（「どれでもない」）を入れるとその 1 文字ぶん減って 25 個です。`N_PER_ROUND` で変更できます。

**2. Ollama の `top_logprobs` は上限 20。** 26 ラベルすべての対数確率を 1 回で受け取ることはできません
（実測: 21 以上を指定すると `top_logprobs must be between 0 and 20` エラー）。
返ってこなかったラベルは下限値で埋めます。確率が 20 位に入らないラベルは元々ほぼ無視できる
大きさなので実害は小さいものの、**埋めた個数は毎回記録して ⑥ で表示します**。
ラベルの割り当ては毎回シャッフルし、特定の遺伝子が固定的に不利にならないようにします。

**3. 思考モデルは、そのままだと 1 文字を答えません。** qwen3 系は最初のトークンとして
`<think>` を出すため、A〜Z の対数確率が 1 つも取れません（実測で 0/26）。
これは黙って全部が下限値になり、**一見動いているのに中身が空**という壊れ方をします。
④ の指示文・few-shot と ⑤ の `think: false` は、この 3 つが揃って初めて letter が返るための
必須の部品です。外さないでください。⑤ の動作確認セルで、実際に返ったトークンを毎回表示します。
**4. 規模がそのまま時間になります。** 費用は生成ではなくプロンプト評価が決めます。
生成するのは常に 1 トークンだけで、選択肢を増やすとプロンプトが伸びるぶんだけ遅くなります
（実測 qwen3:14b: 4 択 0.60s / 133 tok、10 択 0.90s / 177 tok、26 択 1.85s / 279 tok）。
繰り返し回数に対しては完全に線形で、60 回連続でも 1 回あたりは伸びませんでした。
ただし同じ条件でも時間帯によって 1.85s と 3.65s の両方を観測しています。
そのため ⑥ は定数を使わず、その場で 3 回測ってから見積りを出します。
候補 10,000 個・1 群 26 個なら、全員 1 周で 385 呼び出し、各 10 回登場で 3,850 呼び出しです。


## 0. 準備


In [1]:
import os, json, random, math, string, statistics, urllib.request, urllib.error
from collections import defaultdict, Counter

try:
    import pandas as pd
    HAVE_PANDAS = True
except ImportError:
    HAVE_PANDAS = False
    print("pandas がありません:  pip install pandas （表は簡易表示になります）")

OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
print("Ollama:", OLLAMA_HOST)


Ollama: http://localhost:11434


## ① 疾患名


In [2]:
DISEASE = "schizophrenia"      # ← ここに疾患名を書く

print("疾患:", DISEASE)


疾患: schizophrenia


## ② モデルを選ぶ

ローカルの Ollama に入っているモデルを一覧します。`MODEL` に名前を入れてください。
空のままなら一覧の先頭を使います。


In [3]:
def ollama_get(path):
    with urllib.request.urlopen(OLLAMA_HOST + path, timeout=10) as r:
        return json.loads(r.read().decode())


def list_local_models():
    """Ollama に入っているモデル名の一覧。埋め込み専用モデルは採点に使えないので外す。"""
    try:
        models = ollama_get("/api/tags").get("models", [])
    except urllib.error.URLError as e:
        print(f"Ollama に接続できません（{e.reason}）。`ollama serve` は動いていますか。")
        return []
    out = []
    for m in models:
        name = m["name"]
        if any(k in name.lower() for k in ("embed", "bge-", "e5-", "gte-")):
            continue          # 埋め込みモデルは logprobs を返さない
        out.append({"name": name,
                    "size_gb": round(m.get("size", 0) / 1e9, 1),
                    "params": m.get("details", {}).get("parameter_size", "?")})
    return out


AVAILABLE = list_local_models()
for i, m in enumerate(AVAILABLE):
    print(f"  [{i}] {m["name"]:<30}{m["params"]:>8}  {m["size_gb"]}GB")
if not AVAILABLE:
    print("  （使えるモデルが見つかりません）")


  [0] qwen2.5:7b                        7.6B  4.7GB
  [1] qwen3:14b                        14.8B  9.3GB
  [2] cniongolo/biomistral:latest         7B  4.4GB
  [3] gemma3:4b-it-qat                  4.3B  4.0GB
  [4] deepseek-r1:8b                    8.2B  5.2GB
  [5] qwen3:8b                          8.2B  5.2GB
  [6] qwen2.5:14b                      14.8B  9.0GB
  [7] llama3.1:latest                   8.0B  4.9GB


In [15]:
MODEL = "qwen3:14b"       # ← 例: "qwen3:14b"。空なら一覧の先頭

if not MODEL and AVAILABLE:
    MODEL = AVAILABLE[0]["name"]
names = [m["name"] for m in AVAILABLE]
if MODEL and names and MODEL not in names:
    print(f"⚠ {MODEL} は一覧にありません。`ollama pull {MODEL}` が要るかもしれません。")
print("モデル:", MODEL or "★未選択")


モデル: qwen3:14b


## ③ 遺伝子リストを読み込む


In [16]:
GENE_FILE = "genelist01_10000.txt"    # 小さく試すなら "genelist01_500.txt"


def load_genes(path):
    """1 行 1 記号。# で始まる行と空行は読み飛ばす。重複は順序を保って落とす。"""
    seen, out, dupes = set(), [], 0
    with open(path) as f:
        for line in f:
            g = line.strip()
            if not g or g.startswith("#"):
                continue
            if g in seen:
                dupes += 1
                continue
            seen.add(g)
            out.append(g)
    return out, dupes


GENES, n_dupes = load_genes(GENE_FILE)
print(f"{GENE_FILE}: {len(GENES)} 遺伝子" + (f"（重複 {n_dupes} 件を除外）" if n_dupes else ""))
print("先頭:", GENES[:8])


genelist01_500.txt: 504 遺伝子
先頭: ['A2M', 'AAAS', 'ABCA9', 'ABCB11', 'ABHD4', 'ABT1', 'ADAM18', 'ADAMTSL4']


## ④ プロンプト

モデルには **1 文字のラベルだけ**を答えさせ、遺伝子記号そのものは生成させません。
記号を書かせると GPR52 と GPR56 のような似た記号を取り違えるので、
選択肢は記号で提示し、対応付けはコード側で持ちます。

**問い方は「治療標的として最も重要な遺伝子」です。** 「最も強く関連する遺伝子」と聞くと、
GWAS 文献に頻出する遺伝子が勝ち、承認薬の標的が沈みます。実測値は ⑪ に置いてあります。

指示文と few-shot は飾りではありません。26 択という長い問いを前にすると、
モデルは 1 文字ではなく散文（`Answer`, `For`, `It`…）を書き始めます。
実測では、指示文だけ・few-shot だけでは足りず、両方に ⑤ の `think: false` を加えて
初めて letter が返りました。


In [17]:
LETTERS = list(string.ascii_uppercase)      # A..Z

USE_EXIT     = False   # True にすると最後の 1 文字が「どれでもない」になる
N_PER_ROUND  = 26      # 1 群あたりの遺伝子数。上限は 26（USE_EXIT なら 25）

MAX_GENES = len(LETTERS) - (1 if USE_EXIT else 0)
if N_PER_ROUND > MAX_GENES:
    print(f"⚠ N_PER_ROUND={N_PER_ROUND} は上限 {MAX_GENES} を超えるので {MAX_GENES} に切り下げます。")
    N_PER_ROUND = MAX_GENES

INSTRUCTION = "Answer with a single letter only.\n\n"

# 問い方は「関連の強さ」ではなく「治療標的としての重要性」。
# 前者だと GWAS 文献に頻出する遺伝子が勝ち、承認薬の標的が沈む（実測は ⑪ 参照）。
QUESTION = "Most important therapeutic target:"

# few-shot は承認薬とその標的で揃える。どちらも ChEMBL の作用機序で裏を取ったペア。
#   Cystic fibrosis / CFTR      ivacaftor (ACTIVATOR)
#   Rheumatoid arthritis / TNF  adalimumab (INHIBITOR)
# 正解は B と C。特定の文字に寄せないよう散らしてある。
FEWSHOT = (
    "Disease: Cystic fibrosis\n"
    f"{QUESTION}\nA. HBB\nB. CFTR\nC. GPR52\nD. APOE\n"
    "Answer: B\n\n"
    "Disease: Rheumatoid arthritis\n"
    f"{QUESTION}\nA. CFTR\nB. HBB\nC. TNF\nD. GPR56\n"
    "Answer: C\n\n"
)


def build_prompt(disease, genes, use_exit=USE_EXIT, fewshot=True):
    """1 群 → プロンプト文字列と {ラベル: 遺伝子} の対応表。

    対応表を返すのが肝心なところ。ラベルは呼ぶたびに違う遺伝子を指すので、
    「A が正解」ではなく「A が指していたもの」を必ずこの表から読み戻すこと。"""
    head = INSTRUCTION + (FEWSHOT if fewshot else "")
    lines = [f"Disease: {disease}", QUESTION]
    mapping = {}
    for lab, g in zip(LETTERS, genes):
        lines.append(f"{lab}. {g}")
        mapping[lab] = g
    if use_exit:
        exit_lab = LETTERS[len(genes)]
        lines.append(f"{exit_lab}. None of the above")
        mapping[exit_lab] = None
    lines.append("Answer:")
    return head + "\n".join(lines), mapping


demo_prompt, demo_map = build_prompt(DISEASE, GENES[:N_PER_ROUND])
print(demo_prompt)


Answer with a single letter only.

Disease: Cystic fibrosis
Options:
A. HBB
B. CFTR
C. GPR52
D. APOE
Answer: B

Disease: Sickle cell disease
Options:
A. CFTR
B. APOE
C. HBB
D. GPR56
Answer: C

Disease: schizophrenia
Options:
A. A2M
B. AAAS
C. ABCA9
D. ABCB11
E. ABHD4
F. ABT1
G. ADAM18
H. ADAMTSL4
I. ADD3
J. AFG3L2
K. AGMO
L. AHNAK
M. AIDA
N. AJAP1
O. AKAP7
P. AKT1S1
Q. ANHX
R. ANKRD13A
S. ANKRD30A
T. ANKUB1
U. AP2A1
V. ARFIP2
W. ARHGAP20
X. ARHGEF37
Y. ARHGEF38
Z. ARIH2
Answer:


## ⑤ `ranker()` — 1 群を採点する

モデルに次の 1 トークンだけ生成させ、その位置の候補と対数確率を受け取ります。
ラベル A〜Z の対数確率を softmax で正規化し、遺伝子に付け替えて返します。

返ってこなかったラベルは、観測できた最小値からさらに下げた値で埋めます。
**埋めた個数と、実際に生成されたトークンも一緒に返します。**
生成トークンがラベルでない（`<think>` など）なら、そのラウンドは中身が空です。


In [18]:
def _post(path, payload, timeout=180):
    req = urllib.request.Request(OLLAMA_HOST + path,
                                 data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"},
                                 method="POST")
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode())


def first_token_logprobs(prompt, model, top_logprobs=20, no_think=True):
    """次の 1 トークンの (生成トークン, {トークン: 対数確率})。

    top_logprobs は Ollama の上限が 20。no_think は思考モデル対策で、
    対応していないモデルに送ると弾かれることがあるのでその場合は外して再送する。"""
    payload = {"model": model, "prompt": prompt, "stream": False,
               "options": {"temperature": 1, "num_predict": 1},
               "logprobs": True, "top_logprobs": min(top_logprobs, 20)}
    if no_think:
        payload["think"] = False
    try:
        data = _post("/api/generate", payload)
    except urllib.error.HTTPError as e:
        if no_think:
            payload.pop("think", None)          # think 非対応モデル
            data = _post("/api/generate", payload)
        else:
            raise
    if "error" in data:
        raise RuntimeError(str(data["error"])[:200])
    lp = data.get("logprobs")
    if not lp:
        raise RuntimeError("logprobs が返りません。Ollama v0.12.11 以降が必要です。")
    first = lp[0]
    out = {}
    tok, val = first.get("token"), first.get("logprob")
    if tok is not None and val is not None:
        out[tok.strip()] = val
    for alt in (first.get("top_logprobs") or []):
        if isinstance(alt, dict):
            t, v = alt.get("token"), alt.get("logprob")
            if t is None:                      # {token: logprob} 形式のことがある
                for k, vv in alt.items():
                    out.setdefault(str(k).strip(), vv)
            elif v is not None:
                out.setdefault(str(t).strip(), v)
    return tok, out


def ranker(disease, genes, model, use_exit=USE_EXIT, shuffle_labels=True):
    """遺伝子の一群 → {遺伝子: スコア}。

    スコアはラベル集合の中で正規化した確率（0〜1、合計 1）。
    ラベルの割り当ては既定でシャッフルする。順番を固定すると、
    top-20 に入り損ねる位置の遺伝子が毎回同じになって偏るため。"""
    genes = list(genes)
    if shuffle_labels:
        random.shuffle(genes)
    prompt, mapping = build_prompt(disease, genes, use_exit)
    gen_token, raw = first_token_logprobs(prompt, model)

    labels = list(mapping)
    got = {lab: raw[lab] for lab in labels if lab in raw}
    base = {"n_labels": len(labels), "generated": gen_token,
            "n_missing": len(labels) - len(got)}
    if not got:
        # ラベルが 1 つも返っていない。全部を下限で埋めれば「動いた」ように
        # 見えてしまうので、ここでは空を返して呼び出し側に失敗と分からせる。
        return {**base, "scores": {}, "top": None, "top_prob": None}

    floor = min(got.values()) - 10.0        # 見えなかったラベルは下限に置く
    filled = {lab: got.get(lab, floor) for lab in labels}

    m = max(filled.values())
    exp = {lab: math.exp(v - m) for lab, v in filled.items()}
    z = sum(exp.values())
    prob = {lab: v / z for lab, v in exp.items()}

    scores = {mapping[lab]: p for lab, p in prob.items() if mapping[lab] is not None}
    top_lab = max(prob, key=prob.get)
    return {**base, "scores": scores,
            "top": mapping[top_lab],          # None なら「どれでもない」が 1 位
            "top_prob": prob[top_lab]}


### ラベル位置の偏りについて（実測）

**few-shot の正解をどの文字に置くかが、そのまま結果に出ます。** 無関係な遺伝子だけの群を
60 回投げて、勝ったラベルを数えた結果です（一様なら各ラベル 2.3 回）。

| few-shot の正解 | B が勝った回数 | A が勝った回数 | カイ二乗 |
|---|---|---|---|
| B と B | **34/60** | 0 | 466.9 |
| B と C（現在） | 0/60 | **16/60** | 112.5 |

最初は 2 問とも正解を B に置いていました。すると無関係な群の **6 割で B が勝ちます** —
内容ではなくラベルが勝者を決めていたということです。B と C に散らすとこれは消えます。

**ただし偏り自体は残ります。** 今度は先頭の A が 16/60 を占め、カイ二乗は 112.5
（一様なら約 25）。先頭を選びやすい癖は消えていません。

**それでも順位は歪みません。** `ranker()` が毎ラウンド遺伝子とラベルの対応を振り直すので、
A の席に座る遺伝子は毎回入れ替わります。どの遺伝子も 1/26 の確率で A に座るだけなので、
偏りは系統誤差ではなく分散になります。

### ローテーションは効きません（実測）

同じ群をラベル順を変えて複数回投げれば、群の中で位置の効果を相殺できます。
`rotate_scores()` がそれを実装していますが、**呼び出し予算を固定して比べると差が出ませんでした**。

候補 500 個（うち承認薬の標的 6 個）、予算 300 呼び出し固定:

| ローテーション | ラウンド | 登場/個 | 既知 6 個の順位 |
|---|---|---|---|
| 1 | 300 | 15.6 | 1,2,3,4,5,6 |
| 2 | 150 | 7.8 | 1,2,3,4,5,6 |
| 3 | 100 | 5.2 | 1,2,3,4,5,6 |

3 条件とも承認薬の標的が 1〜6 位を独占し、**完全に同じ結果**でした。
カイ二乗 112.5 の残存偏りは、順位を動かすほどの力を持っていません。

理屈もそのとおりです。ラベルは既に毎ラウンド振り直されているので、ローテーションで減るのは
分散だけ。その分散はラウンドを増やしても同じだけ減り、しかもラウンドを増やすほうは
**対戦相手の組み合わせも増える**ぶん有利です。

実装は残してありますが、既定は `ROTATIONS = 1`（使わない）です。
同じ予算で得をしないと分かっている手段に、後からもう一度時間を使わないために測定値を残します。


In [ ]:
def rotate_scores(disease, genes, model, rotations=1, rng=None):
    """同じ群をラベル順を変えて複数回出題し、確率を平均する。

    ラベルの割り当ては ranker() が毎ラウンド振り直すので、特定の遺伝子が
    systematically 得をすることは元々ない。ここで減るのは系統誤差ではなく分散。
    rotations=1 なら従来と同じ（1 回だけ投げる）。

    巡回シフトを使う。同じ並びを保ったまま位置だけずらすので、
    どの遺伝子も rotations 個の異なる位置を通る。"""
    base = list(genes)
    (rng or random).shuffle(base)
    agg = defaultdict(float)
    ok = 0
    missing = 0
    for r in range(max(1, rotations)):
        shift = (r * len(base)) // max(1, rotations)   # 位置を散らす
        rolled = base[shift:] + base[:shift]
        res = ranker(disease, rolled, model, shuffle_labels=False)
        if not res["scores"]:
            continue
        ok += 1
        missing += res["n_missing"]
        for g, pr in res["scores"].items():
            agg[g] += pr
    if not ok:
        return {"scores": {}, "n_missing": 0, "n_ok": 0}
    return {"scores": {g: v / ok for g, v in agg.items()},
            "n_missing": missing // ok, "n_ok": ok}


In [19]:
# 動作確認（1 群だけ）— ここで letter が返らなければ先へ進んでも無意味
if MODEL:
    random.seed(0)
    probe = ranker(DISEASE, GENES[:N_PER_ROUND], MODEL)
    print(f"生成トークン : {probe['generated']!r}")
    print(f"ラベル取得   : {probe['n_labels'] - probe['n_missing']}/{probe['n_labels']}"
          f"（{probe['n_missing']} 個は下限で補完。Ollama の上限 20 のため 6 個前後は正常）")
    if not probe["scores"]:
        print("\n❌ ラベルが 1 つも返っていません。以下を確認してください:")
        print("   ・思考モデルなら think:false が効いているか（生成トークンが <think> なら効いていない）")
        print("   ・④ の INSTRUCTION と FEWSHOT を消していないか")
        print("   ・別のモデルを試す")
    else:
        print(f"1 位         : {probe['top']}  (p={probe['top_prob']:.3f})")
        top5 = sorted(probe["scores"].items(), key=lambda kv: -kv[1])[:5]
        print("上位 5       :", [(g, round(p, 4)) for g, p in top5])
else:
    print("モデル未選択のため実行しません。")


生成トークン : 'A'
ラベル取得   : 19/26（7 個は下限で補完。Ollama の上限 20 のため 6 個前後は正常）
1 位         : AKAP7  (p=0.991)
上位 5       : [('AKAP7', 0.9909), ('A2M', 0.0063), ('AFG3L2', 0.0013), ('ANKUB1', 0.0007), ('ADD3', 0.0004)]


## ⑥ 群を作って繰り返す

毎回 `N_PER_ROUND` 個を選んで `ranker()` に投げ、遺伝子ごとにスコアを溜めます。

### 全ての遺伝子を同じ回数だけ競争に入れる

毎回独立に無作為抽出すると、登場回数は二項分布になって大きくばらつきます。
501 遺伝子・1 群 26 個・100 ラウンドの実測で、**ランダム抽出では登場回数が 0〜13 回**
（平均 5.2、SD 2.3、一度も出ない遺伝子あり）。13 回出た遺伝子と 1 回しか出ない遺伝子の
平均スコアを同じ表で並べても、比べているのは実力ではなく試行回数です。

`balanced` は **山札方式**でこれを揃えます。全遺伝子を 1 つの山に切って上から 26 枚ずつ配り、
山が尽きたら切り直す。カードゲームで全員に同じ枚数を配るのと同じ理屈です。
同じ条件での実測は **5〜6 回（SD 0.39、未出現ゼロ）**、登場回数の差は常に最大 1 に収まります。

| | 登場回数の幅 | SD | 未出現 |
|---|---|---|---|
| ランダム | 0〜13 回 | 2.27 | 1 遺伝子 |
| 均等（山札） | 5〜6 回 | 0.39 | なし |

同じ群に同じ遺伝子が二度入らないよう、配る際に重複は飛ばします。
山を切り直すたびに相手の組み合わせは変わるので、**誰と当たるか**は依然ランダムです。
揃うのは回数だけで、対戦相手の強さまでは揃いません（そこは繰り返し回数で均します）。


In [20]:
def random_groups(genes, n, rounds, rng):
    """毎回独立に無作為抽出。登場回数は揃わない。比較用。"""
    return [rng.sample(genes, min(n, len(genes))) for _ in range(rounds)]


def balanced_groups(genes, n, rounds, rng):
    """山札方式。全遺伝子を切って上から配り、尽きたら切り直す。

    どの遺伝子も、切り直しごとにちょうど 1 回ずつ配られる。したがって
    登場回数の差はラウンド数によらず最大 1。同一群内の重複だけは飛ばす。
    """
    n = min(n, len(genes))
    deck, out = [], []
    for _ in range(rounds):
        group = []
        while len(group) < n:
            if not deck:
                deck = list(genes)
                rng.shuffle(deck)
            picked = None
            for i, g in enumerate(deck):
                if g not in group:          # 同じ群に二度入れない
                    picked = deck.pop(i)
                    break
            if picked is None:              # 山の残りが全部この群にある
                deck = []
                continue
            group.append(picked)
        out.append(group)
    return out


def appearance_report(groups, genes, label):
    """登場回数が揃っているかを数字で見せる。揃っていないなら順位を読む前に直す。"""
    c = Counter(g for grp in groups for g in grp)
    counts = [c.get(g, 0) for g in genes]
    print(f"  {label:<16} min={min(counts):<3} max={max(counts):<3} "
          f"平均={statistics.mean(counts):.2f}  SD={statistics.pstdev(counts):.2f}  "
          f"未出現={sum(1 for x in counts if x == 0)}")
    return counts


In [21]:
SAMPLING = "balanced"   # "balanced" = 山札方式（既定） / "random" = 毎回無作為
N_ROUNDS = 100          # ← 繰り返し回数
ROTATIONS = 1           # 1 群あたりのラベル順の通り数。呼び出しは ROUNDS×ROTATIONS
SEED     = 0

ROUNDS_PER_EPOCH = -(-len(GENES) // N_PER_ROUND) if GENES else 0
print(f"{len(GENES)} 遺伝子 / 1 群 {N_PER_ROUND} 個")
print(f"全員を 1 周させるのに {ROUNDS_PER_EPOCH} ラウンド")

expected = N_ROUNDS * N_PER_ROUND / len(GENES) if GENES else 0
print(f"{N_ROUNDS} ラウンド → 1 遺伝子あたり平均 {expected:.1f} 回の登場")
if expected < 3:
    print("  ⚠ 少なすぎます。順位の差はほとんど試行回数の偶然です。")
if N_ROUNDS % ROUNDS_PER_EPOCH:
    nxt = (N_ROUNDS // ROUNDS_PER_EPOCH + 1) * ROUNDS_PER_EPOCH
    print(f"  ヒント: {ROUNDS_PER_EPOCH} の倍数（次は {nxt}）にすると全員が完全に同じ回数になります。")

# 所要時間。定数を決め打ちにせず、その場で 3 回測る。
# 同じ機械・同じモデルでも時間帯によって倍近く変わることがあるので、
# 決め打ちの数字は当てにならない（実測で 1.85s と 3.65s の両方を観測している）。
def measure_sec_per_call(n=3):
    """ダミーの群を投げて 1 呼び出しの実時間を測る。1 回目は捨てる。"""
    import time
    rng = random.Random(12345)
    ts = []
    for k in range(n + 1):
        grp = rng.sample(GENES, min(N_PER_ROUND, len(GENES)))
        t0 = time.time()
        try:
            ranker(DISEASE, grp, MODEL)
        except Exception as e:
            print(f"  計測に失敗: {type(e).__name__}: {e}")
            return None
        if k:                      # 1 回目はウォームアップ
            ts.append(time.time() - t0)
    return statistics.mean(ts)


SEC_PER_CALL = None
if MODEL and GENES:
    SEC_PER_CALL = measure_sec_per_call()
    if SEC_PER_CALL:
        print(f"1 呼び出しの実測: {SEC_PER_CALL:.2f}s")
SEC_PER_CALL = SEC_PER_CALL or 2.0     # 測れなかったときの目安
mins = N_ROUNDS * SEC_PER_CALL / 60
print(f"想定所要時間: 約 {mins:.0f} 分" + (f"（{mins/60:.1f} 時間）" if mins >= 90 else ""))
if mins >= 60:
    full = ROUNDS_PER_EPOCH * 10 * SEC_PER_CALL / 3600
    print(f"  ⚠ 長時間かかります。全遺伝子を 10 回ずつ登場させるなら "
          f"{ROUNDS_PER_EPOCH * 10} 呼び出し = 約 {full:.1f} 時間。")
    print(f"  ・まず GENE_FILE を小さいリストにして通しで動くか確かめてください。")
    print(f"  ・N_PER_ROUND を減らすと 1 回は速くなりますが、必要ラウンド数は増えます。")

# 実際にモデルへ投げる前に、群の作り方だけ確認しておく
if GENES:
    print("\n登場回数の分布（モデル呼び出しなし・シミュレーション）:")
    for name, fn in (("random", random_groups), ("balanced", balanced_groups)):
        appearance_report(fn(GENES, N_PER_ROUND, N_ROUNDS, random.Random(SEED)),
                          GENES, name)


504 遺伝子 / 1 群 26 個
全員を 1 周させるのに 20 ラウンド
100 ラウンド → 1 遺伝子あたり平均 5.2 回の登場

登場回数の分布（モデル呼び出しなし・シミュレーション）:
  random           min=0   max=13  平均=5.16  SD=2.26  未出現=1
  balanced         min=5   max=6   平均=5.16  SD=0.37  未出現=0


In [22]:
def run_rounds(disease, genes, model, n_rounds, n_per_round, seed=0,
               sampling="balanced", rotations=1, progress_every=10):
    """繰り返して {遺伝子: [スコア,...]} を集める。

    群の作り方を先に全部決めてから回す。こうしておくと、途中で失敗した回が
    あっても「どの遺伝子が何回出題されたはずか」が後から分かる。"""
    rng = random.Random(seed)
    make = balanced_groups if sampling == "balanced" else random_groups
    groups = make(genes, n_per_round, n_rounds, rng)

    collected = defaultdict(list)
    wins = defaultdict(int)
    missing, none_wins, failures, empty = [], 0, 0, 0
    matches = []

    for i, group in enumerate(groups):
        try:
            res = rotate_scores(disease, group, model, rotations, rng)
        except Exception as e:
            failures += 1
            print(f"  round {i}: 失敗 {type(e).__name__}: {e}")
            continue
        if not res["scores"]:
            empty += 1
            continue
        for g, p in res["scores"].items():
            collected[g].append(p)
        matches.append((tuple(group), dict(res["scores"])))
        best = max(res["scores"], key=res["scores"].get)
        wins[best] += 1
        missing.append(res["n_missing"])
        if progress_every and (i + 1) % progress_every == 0:
            print(f"  {i + 1}/{n_rounds} 回", flush=True)

    return {"collected": collected, "wins": wins, "missing": missing,
            "none_wins": none_wins, "failures": failures, "empty": empty,
            "groups": groups, "sampling": sampling, "matches": matches}


run = None
if MODEL and GENES:
    run = run_rounds(DISEASE, GENES, MODEL, N_ROUNDS, N_PER_ROUND, SEED,
                     sampling=SAMPLING, rotations=ROTATIONS)
    print(f"\n完了（{run['sampling']}）。通信失敗 {run['failures']} 回 / "
          f"ラベルが取れなかった回 {run['empty']} 回")
    if run["missing"]:
        print(f"補完したラベル数: 中央値 {statistics.median(run['missing']):.0f} / "
              f"{N_PER_ROUND + (1 if USE_EXIT else 0)}")
    if USE_EXIT:
        print(f"「どれでもない」が 1 位だった回: {run['none_wins']}/{N_ROUNDS}")

    # 出題された回数と、実際に採点できた回数は失敗のぶんだけずれる
    asked = Counter(g for grp in run["groups"] for g in grp)
    scored = {g: len(v) for g, v in run["collected"].items()}
    print(f"出題された遺伝子: {len(asked)}/{len(GENES)}  "
          f"（出題回数 {min(asked.values())}〜{max(asked.values())} 回）")
    print(f"採点された遺伝子: {len(scored)}/{len(GENES)}")
else:
    print("モデル未選択、または遺伝子リストが空です。")


  10/100 回
  20/100 回
  30/100 回
  40/100 回
  50/100 回
  60/100 回
  70/100 回
  80/100 回
  90/100 回
  100/100 回

完了（balanced）。通信失敗 0 回 / ラベルが取れなかった回 0 回
補完したラベル数: 中央値 6 / 26
出題された遺伝子: 504/504  （出題回数 5〜6 回）
採点された遺伝子: 504/504


## ⑦ 遺伝子ごとのスコア


In [23]:
def to_rows(run, genes):
    """平均スコアの降順。ばらつきと登場回数を必ず併記する。

    平均だけを見ると、2 回しか出ていない遺伝子と 30 回出ている遺伝子が
    同じ確からしさに見えてしまう。"""
    rows = []
    for g in genes:
        vals = run["collected"].get(g, [])
        if not vals:
            rows.append({"gene": g, "mean_score": None, "sd": None,
                         "n_seen": 0, "n_wins": 0, "max_score": None})
            continue
        rows.append({
            "gene": g,
            "mean_score": statistics.mean(vals),
            "sd": statistics.stdev(vals) if len(vals) > 1 else 0.0,
            "n_seen": len(vals),
            "n_wins": run["wins"].get(g, 0),
            "max_score": max(vals),
        })
    rows.sort(key=lambda r: (r["mean_score"] is None, -(r["mean_score"] or 0)))
    for i, r in enumerate(rows, 1):
        r["rank"] = i
    return rows


rows = to_rows(run, GENES) if run else []
df = None

if rows and HAVE_PANDAS:
    df = pd.DataFrame(rows)[
        ["rank", "gene", "mean_score", "sd", "n_seen", "n_wins", "max_score"]]
    df = df.round({"mean_score": 5, "sd": 5, "max_score": 5})
    try:
        display(df.head(30))
    except NameError:
        print(df.head(30).to_string(index=False))
elif rows:
    for r in rows[:30]:
        ms = "-" if r["mean_score"] is None else f"{r['mean_score']:.5f}"
        print(f"{r['rank']:>4}  {r['gene']:<12}{ms:>10}  "
              f"n={r['n_seen']:<4}wins={r['n_wins']}")
else:
    print("結果がありません。")


,rank,gene,mean_score,sd,n_seen,n_wins,max_score
0,1,DRD2,0.91017,0.18849,5,5,0.99880
1,2,SETD5,0.85381,0.27552,5,5,1.00000
2,3,TCF4,0.80119,0.44338,5,4,1.00000
3,4,CSMD1,0.80055,0.44540,5,4,1.00000
4,5,KMT2D,0.78495,0.43902,5,4,0.99194
5,6,ZSWIM3,0.40339,0.54435,5,2,0.99996
6,7,STRIP1,0.39044,0.50098,5,2,0.97832
7,8,PRELID1,0.37494,0.51519,5,2,0.99818
8,9,HNRNPF,0.36864,0.48679,5,2,0.91561
9,10,TGIF1,0.32051,0.40676,5,1,0.99370


### 保存


In [24]:
import csv as _csv

if rows:
    out = f"{DISEASE.replace(' ', '_')}_scores.csv"
    if df is not None:
        df.to_csv(out, index=False)
    else:
        with open(out, "w", newline="") as fh:
            cols = ["rank", "gene", "mean_score", "sd", "n_seen", "n_wins", "max_score"]
            w = _csv.DictWriter(fh, fieldnames=cols, extrasaction="ignore")
            w.writeheader(); w.writerows(rows)
    print("書き出しました:", out)


書き出しました: schizophrenia_scores.csv


## ⑧ トーナメント方式（勝ち抜き戦）

⑥⑦ は全遺伝子に平均スコアを付ける総当たり的なやり方です。上位だけが要るなら高すぎます。
10,000 遺伝子で各 10 回登場させると 3,850 呼び出し（約 2 時間）かかりますが、
そこまでしても 1 つの遺伝子が見た相手は全体の 2.6% にすぎません。

勝ち抜き戦なら、1 回戦で 385 試合、勝った 385 個で 2 回戦 15 試合、その勝者 15 個で決勝 1 試合。
**合計 401 試合（約 12 分）** で決着します。しかも勝者同士を直接戦わせているぶん、
「たまたま弱い群にいた」遺伝子が上位に残りにくくなります。

### 指定できる数

| 変数 | 意味 |
|---|---|
| `TOURNEY_GROUP_SIZE` | 1 試合に出す遺伝子の数（上限 26） |
| `TOURNEY_ADVANCE` | 1 試合から勝ち上がる数。2 以上にすると取りこぼしが減るが試合数は増える |
| `TOURNEY_TARGET` | 何個まで絞ったら決勝にするか |
| `TOURNEY_REPEATS` | 1 試合を何回投げて平均するか。ラベル順の偶然で敗退するのを防ぐ |

### この方式の弱点

**敗者にはスコアが付きません。** 1 回戦で優勝候補と同じ組になった遺伝子は、2 位の実力でも
そこで消えます。トーナメントが答えるのは「どれが勝ち残るか」であって「全遺伝子の順位」ではありません。
得られる順位情報は **何回戦まで残ったか** だけです。

取りこぼしが気になるなら `TOURNEY_ADVANCE` を 2〜3 に上げてください。試合数は増えますが、
それでも総当たり平均よりはるかに安く済みます。


In [ ]:
def tournament_cost(n, group_size, advance, target, repeats):
    """実際に投げる前に試合数を数える。無限ループの検出も兼ねる。"""
    if advance >= group_size:
        raise ValueError("TOURNEY_ADVANCE は TOURNEY_GROUP_SIZE より小さくすること"
                         "（同数以上だと誰も脱落せず終わらない）")
    alive, calls, rounds = n, 0, 0
    while alive > target:
        n_groups = -(-alive // group_size)
        calls += n_groups * repeats
        nxt = n_groups * advance
        if nxt >= alive:            # 縮まないなら打ち切り
            break
        alive, rounds = nxt, rounds + 1
    calls += repeats                # 決勝
    return calls, rounds + 1, alive


def tournament(disease, genes, model, group_size=26, advance=1, target=26,
               repeats=1, seed=0, verbose=True):
    """勝ち抜き戦。返すのは「何回戦まで残ったか」と決勝の順位。

    各試合は ranker() をそのまま使う（ラベル順は ranker 側でシャッフルされる）。
    repeats > 1 なら同じ組を複数回投げ、確率を足し合わせてから勝者を決める。
    ラベルの割り当てが毎回変わるので、これは位置バイアスに対する多数決になる。
    """
    rng = random.Random(seed)
    alive = list(genes)
    rng.shuffle(alive)

    exit_round = {g: None for g in genes}   # None = 決勝まで残った
    best_score = defaultdict(float)
    n_matches = defaultdict(int)
    calls = failures = 0
    rnd = 0

    while len(alive) > target:
        rnd += 1
        groups = [alive[i:i + group_size]
                  for i in range(0, len(alive), group_size)]
        if len(groups) > 1 and len(groups[-1]) < 2:
            groups[-2].extend(groups.pop())     # 端数 1 個は前の組に吸収

        winners = []
        for grp in groups:
            agg = defaultdict(float)
            ok = 0
            for _ in range(repeats):
                try:
                    res = ranker(disease, grp, model)
                    calls += 1
                except Exception as e:
                    failures += 1
                    print(f"  R{rnd} 試合失敗: {type(e).__name__}: {e}")
                    continue
                if not res["scores"]:
                    failures += 1
                    continue
                ok += 1
                for g, p in res["scores"].items():
                    agg[g] += p / repeats
            for g in grp:
                n_matches[g] += 1
            if not ok:
                # 情報が得られなかった組は誰も落とさない。適当に落とすより
                # 試合数が増えるほうがまし。
                winners.extend(grp)
                continue
            for g, s in agg.items():
                best_score[g] = max(best_score[g], s)
            ranked = sorted(agg, key=lambda g: -agg[g])
            winners.extend(ranked[:min(advance, len(ranked))])

        for g in alive:
            if g not in winners:
                exit_round[g] = rnd
        if verbose:
            print(f"  {rnd} 回戦: {len(alive):>6} → {len(winners):>6}  "
                  f"（{len(groups)} 試合 × {repeats}）", flush=True)
        if len(winners) >= len(alive):
            print("  ⚠ 絞り込めませんでした。TOURNEY_ADVANCE を小さくしてください。")
            break
        alive = winners
        rng.shuffle(alive)

    # 決勝
    final = []
    if alive:
        agg = defaultdict(float)
        ok = 0
        for _ in range(repeats):
            try:
                res = ranker(disease, alive[:group_size], model)
                calls += 1
            except Exception as e:
                failures += 1
                print(f"  決勝失敗: {type(e).__name__}: {e}")
                continue
            if not res["scores"]:
                failures += 1
                continue
            ok += 1
            for g, p in res["scores"].items():
                agg[g] += p / repeats
        for g in alive:
            n_matches[g] += 1
        final = sorted(agg, key=lambda g: -agg[g])
        for g, s in agg.items():
            best_score[g] = max(best_score[g], s)
        if verbose:
            print(f"  決勝: {len(alive)} 個 → 優勝 {final[0] if final else '（判定できず'}")

    return {"exit_round": exit_round, "final": final, "best_score": best_score,
            "n_matches": n_matches, "calls": calls, "failures": failures,
            "rounds": rnd + 1}


In [ ]:
TOURNEY_GROUP_SIZE = 26   # 1 試合に出す数（上限 26）
TOURNEY_ADVANCE    = 1    # 1 試合から勝ち上がる数
TOURNEY_TARGET     = 26   # 何個まで絞ったら決勝にするか
TOURNEY_REPEATS    = 1    # 1 試合あたりの投げる回数
TOURNEY_SEED       = 0

if GENES:
    calls, rounds, finalists = tournament_cost(
        len(GENES), TOURNEY_GROUP_SIZE, TOURNEY_ADVANCE,
        TOURNEY_TARGET, TOURNEY_REPEATS)
    print(f"{len(GENES)} 遺伝子 → {rounds} 回戦、合計 {calls} 試合")
    print(f"想定所要時間: 約 {calls * SEC_PER_CALL / 60:.0f} 分")
    flat = -(-len(GENES) // N_PER_ROUND) * 10
    print(f"（参考: ⑥ の総当たり平均で各 10 回登場させると {flat} 呼び出し = "
          f"約 {flat * SEC_PER_CALL / 60:.0f} 分）")


In [ ]:
tour = None
if MODEL and GENES:
    tour = tournament(DISEASE, GENES, MODEL,
                      group_size=TOURNEY_GROUP_SIZE,
                      advance=TOURNEY_ADVANCE,
                      target=TOURNEY_TARGET,
                      repeats=TOURNEY_REPEATS,
                      seed=TOURNEY_SEED)
    print(f"\n実際の試合数 {tour['calls']}  失敗 {tour['failures']}")
else:
    print("モデル未選択のため実行しません。")


### 結果

`exit_round` が大きいほど長く勝ち残ったということです。決勝進出者は `final_rank` を持ちます。
**1 回戦敗退の遺伝子どうしには順位が付きません** — 同じ「1 回戦敗退」でも、優勝候補と
当たって消えたのか本当に弱いのかは、この方式では区別できません。


In [ ]:
def tournament_rows(tour, genes):
    if not tour:
        return []
    final_rank = {g: i + 1 for i, g in enumerate(tour["final"])}
    rows = []
    for g in genes:
        er = tour["exit_round"].get(g)
        rows.append({
            "gene": g,
            "exit_round": "決勝" if er is None else er,
            "final_rank": final_rank.get(g),
            "best_score": round(tour["best_score"].get(g, 0.0), 5),
            "n_matches": tour["n_matches"].get(g, 0),
        })
    # 決勝進出者を上に、その中は決勝順位。以下は敗退ラウンドの遅い順
    rows.sort(key=lambda r: (
        0 if r["final_rank"] else 1,
        r["final_rank"] or 0,
        -(r["exit_round"] if isinstance(r["exit_round"], int) else 99),
        -r["best_score"]))
    for i, r in enumerate(rows, 1):
        r["rank"] = i
    return rows


trows = tournament_rows(tour, GENES)
if trows and HAVE_PANDAS:
    tdf = pd.DataFrame(trows)[
        ["rank", "gene", "exit_round", "final_rank", "best_score", "n_matches"]]
    try:
        display(tdf.head(30))
    except NameError:
        print(tdf.head(30).to_string(index=False))
elif trows:
    for r in trows[:30]:
        print(f"{r['rank']:>4}  {r['gene']:<12}敗退={r['exit_round']}  "
              f"決勝順位={r['final_rank']}  score={r['best_score']}")
else:
    print("結果がありません。")

if trows:
    from collections import Counter as _C
    dist = _C(r["exit_round"] for r in trows)
    print("\n敗退ラウンドの分布:", dict(sorted(dist.items(), key=lambda kv: str(kv[0]))))


## ⑨ 段階的な絞り込み

⑧ の一発勝負では 10,000 個中 9,615 個が 1 回戦敗退で区別できませんでした。
ここでは負けても消えません。各段階で全員に平均スコアが付き、下位を切るだけです。
落ちた遺伝子にもスコアが残るので、全個体が順位付けされます。

```
第 1 段階  10,000 × 2,000 ラウンド → 各 5.2 回登場 → 上位 1,000 通過
第 2 段階   1,000 ×   500 ラウンド → 各  13 回登場 → 上位 200 通過
第 3 段階     200 ×   200 ラウンド → 各  26 回登場 → 最終順位
```

### 承知しておくべき弱点

スコアは**群の中の相対確率**です。無関係な遺伝子だけの群でも勝者には p≈0.95 が付きます
（実測: 15 回投げて 15 回とも誰かが勝つ、1 位の確率の中央値 0.947）。
本物が自分の群で勝った 1.0 と、弱い群で勝っただけの偽物の 1.0 は、**同じ値**です。

10,000 個・26 個ずつなら 1 周で 385 個の勝者が出ます。5 周すれば全勝する遺伝子が数百個生まれ、
その中の順序は実質的に任意です。**上位数十位の並びを実力差として読まないでください。**

繰り返し回数を増やしても、この同点は解消しません。分散ではなく、
「誰と戦って勝ったか」を見ていないことによるものだからです。


In [ ]:
def winnow_preview(n_genes, stages, group_size, sec_per_call):
    """実行前に、各段階の登場回数・呼び出し数・時間を出す。"""
    pool, total = n_genes, 0
    print(f"{'段階':<5}{'対象':>9}{'ラウンド':>10}{'登場/個':>10}{'時間':>9}")
    for si, (rounds, keep) in enumerate(stages, 1):
        appear = rounds * group_size / pool if pool else 0
        total += rounds
        print(f"{si:<5}{pool:>9}{rounds:>10}{appear:>10.1f}{rounds * sec_per_call / 60:>8.0f}分")
        if appear < 3:
            print(f"    ⚠ 登場 {appear:.1f} 回では、切る根拠が試行回数の偶然になります。")
        pool = min(keep, pool) if keep else pool
    print(f"{'合計':<5}{'':>9}{total:>10}{'':>10}{total * sec_per_call / 60:>8.0f}分")
    return total


def winnow(disease, genes, model, stages, group_size=26, seed=0,
           rotations=1, progress_every=100, verbose=True):
    """段階的に絞り込む。stages = [(ラウンド数, 通過数), ...]。

    各段階で現在のプールを均等配りで出題し、平均スコアの上位だけを次段階へ送る。
    段階が進むほど相手が強くなるので、**段階をまたいでスコアを比べてはいけない**。
    順位は「到達段階」を第一キーにする。"""
    pool = list(genes)
    record = {}
    calls = failures = 0
    matches = []

    for si, (rounds, keep) in enumerate(stages, 1):
        rng = random.Random(seed + si)
        groups = balanced_groups(pool, group_size, rounds, rng)
        collected = defaultdict(list)

        for i, grp in enumerate(groups):
            try:
                res = rotate_scores(disease, grp, model, rotations, rng)
                calls += res["n_ok"]
            except Exception as e:
                failures += 1
                print(f"  S{si} round {i}: 失敗 {type(e).__name__}: {e}")
                continue
            if not res["scores"]:
                failures += 1
                continue
            for g, pr in res["scores"].items():
                collected[g].append(pr)
            matches.append((tuple(grp), dict(res["scores"])))
            if progress_every and (i + 1) % progress_every == 0 and verbose:
                print(f"    S{si}: {i + 1}/{len(groups)}", flush=True)

        means = {g: statistics.mean(v) for g, v in collected.items()}
        for g in pool:
            record[g] = {"stage": si, "mean": means.get(g),
                         "n": len(collected.get(g, []))}
        ranked = sorted(pool,
                        key=lambda g: -(means[g] if g in means else float("-inf")))
        nxt = ranked[:keep] if keep else ranked
        if verbose:
            print(f"  第 {si} 段階: {len(pool):>6} → {len(nxt):>6}  "
                  f"（{len(groups)} ラウンド、採点済み {len(means)}）", flush=True)
        pool = nxt

    return {"record": record, "final_order": pool, "calls": calls,
            "failures": failures, "n_stages": len(stages), "matches": matches}


In [ ]:
# (ラウンド数, 通過数)。通過数 None は全員残す＝最終段階
STAGES = [
    (2000, 1000),
    (500,   200),
    (200,  None),
]
WINNOW_SEED = 0
WINNOW_ROTATIONS = 1   # 1 群あたりのラベル順。呼び出しは ラウンド×これ

if GENES:
    tot = winnow_preview(len(GENES), STAGES, N_PER_ROUND, SEC_PER_CALL)
    if WINNOW_ROTATIONS > 1:
        print(f"  ローテーション {WINNOW_ROTATIONS} 通りなので実際は "
              f"{tot * WINNOW_ROTATIONS} 呼び出し = "
              f"約 {tot * WINNOW_ROTATIONS * SEC_PER_CALL / 60:.0f} 分")


In [ ]:
win = None
if MODEL and GENES:
    win = winnow(DISEASE, GENES, MODEL, STAGES,
                 group_size=N_PER_ROUND, seed=WINNOW_SEED,
                 rotations=WINNOW_ROTATIONS)
    print(f"\n実際の呼び出し {win['calls']}  失敗 {win['failures']}")
else:
    print("モデル未選択のため実行しません。")


### 結果

`stage` は何段階目まで進んだかです。段階が上がるほど相手が強くなるので、
**`mean_score` を段階をまたいで比べてはいけません**。第 1 段階で落ちた遺伝子の 0.9 より、
最終段階を勝ち抜いた遺伝子の 0.4 のほうが強いことになります。

同じ段階の中でも、上位は同点が多く並びます。`mean_score` が 1.0 付近に固まっている範囲は、
順序に意味がないと考えてください。


In [ ]:
def winnow_rows(win, genes):
    if not win:
        return []
    order = {g: i for i, g in enumerate(win["final_order"])}
    rows = []
    for g in genes:
        r = win["record"].get(g, {})
        rows.append({
            "gene": g,
            "stage": r.get("stage"),
            "mean_score": None if r.get("mean") is None else round(r["mean"], 5),
            "n_seen": r.get("n", 0),
            "_o": order.get(g, 10 ** 9),
        })
    # 到達段階が深い順 → 最終段階内は順位 → それ以外は最後のスコア順
    rows.sort(key=lambda r: (-(r["stage"] or 0), r["_o"],
                            -(r["mean_score"] if r["mean_score"] is not None else -1)))
    for i, r in enumerate(rows, 1):
        r["rank"] = i
        r.pop("_o")
    return rows


wrows = winnow_rows(win, GENES)
if wrows and HAVE_PANDAS:
    wdf = pd.DataFrame(wrows)[["rank", "gene", "stage", "mean_score", "n_seen"]]
    try:
        display(wdf.head(30))
    except NameError:
        print(wdf.head(30).to_string(index=False))
elif wrows:
    for r in wrows[:30]:
        print(f"{r['rank']:>5}  {r['gene']:<12}S{r['stage']}  "
              f"{r['mean_score']}  n={r['n_seen']}")
else:
    print("結果がありません。")

if wrows:
    from collections import Counter as _C2
    print("\n到達段階の分布:", dict(sorted(_C2(r["stage"] for r in wrows).items())))
    ns = [r["n_seen"] for r in wrows if r["stage"] == 1]
    if ns:
        print(f"第 1 段階の登場回数: {min(ns)}〜{max(ns)} 回")


## ⑩ Bradley-Terry レーティング

平均スコアには構造的な欠陥があります。**誰と戦って勝ったかを見ていません。**
弱い 25 個に勝った 1.0 と、DRD2 級に勝った 1.0 が同じ重みになります。
10,000 個・26 個ずつなら 1 周で 385 個の勝者が出るので、数周まわせば全勝する遺伝子が
数百個生まれ、その中の順序は任意です。上位数十位の並びが実力差として読めないのはこのためです。

Bradley-Terry（Elo と同じ族）はここを直します。1 回の出題（26 個中どれが選ばれるか）を
**強さ π の比による選択**とみなし、全ラウンドをまとめて最尤推定します。

```
P(群 G で i が選ばれる) = π_i / Σ_{j∈G} π_j
```

強い相手に勝てば π は大きく上がり、弱い相手にだけ勝っても上がりません。

**一度も勝てなかった遺伝子は、依然として同点のまま並びます。** これは欠陥ではなく正しい挙動です。
全敗どうしを区別する情報はどこにも無いので、順序を付ければそれは捏造になります。
実行結果の「同点の遺伝子」の数がそのまま、**証拠が足りていない遺伝子の数**です。
この数を減らしたければラウンドを増やすしかありません。

**追加の推論コストはゼロです。** すでに集めた出題記録を集計し直すだけなので、
⑥ でも ⑨ でも、実行済みの結果にそのまま適用できます。

### 正しさの確認

真の強さが分かっている合成データ（500 個・2,000 ラウンド）で検証しました。

| | 真の強さとの Spearman | 同点 | 真の上位 10 の推定順位 |
|---|---|---|---|
| BT | **+0.9999** | 0 個 | 1,2,3,4,5,6,7,8,9,10 |
| 平均スコア | +0.9993 | 2 個 | 1〜9 と **13** |

（この合成データでは全遺伝子が 4 回以上勝っているため同点が 0 になります。
実データで登場回数が少ないと、勝てなかった遺伝子が大量に同点で並びます。）

### 直らないもの

BT は集計方法であって、モデルの知識は変えません。**負けを勝ちに変えることはできません。**
実測で DRD2 の勝率は 13/20（無作為 25 個相手）で、偽の勝者の平均スコア 0.857 を下回ります。
この場合、BT を通しても DRD2 は上位に来ません。順位付けの問題ではなく、
モデルがこの疾患について持っている知識の形の問題です。


In [ ]:
def bradley_terry(matches, alpha=0.5, iters=300, tol=1e-9):
    """Luce/BT を MM 法で最尤推定する。

    matches は (群の遺伝子, {遺伝子: 確率}) の列。観測は「1 個が選ばれた」ではなく
    確率ベクトルなので、勝ち数は小数で数える（情報を捨てないため）。

    alpha は強さ 1 の仮想対戦相手との引き分けで、一度も勝てなかった遺伝子の
    レーティングが -inf に飛ぶのを防ぐ。0 にすると全敗の遺伝子が発散する。
    """
    W = defaultdict(float)
    sets = defaultdict(list)
    for grp, probs in matches:
        for g in grp:
            W[g] += probs.get(g, 0.0)
            sets[g].append(grp)
    if not W:
        return {}

    pi = {g: 1.0 for g in W}
    for _ in range(iters):
        cache = {}
        new = {}
        for g in pi:
            d = 0.0
            for grp in sets[g]:
                s = cache.get(id(grp))
                if s is None:
                    s = sum(pi[x] for x in grp if x in pi)
                    cache[id(grp)] = s
                if s > 0:
                    d += 1.0 / s
            d += 2 * alpha / (pi[g] + 1.0)
            new[g] = (W[g] + alpha) / d if d > 0 else pi[g]
        gm = math.exp(statistics.mean(math.log(v) for v in new.values() if v > 0))
        new = {g: v / gm for g, v in new.items()}
        delta = max(abs(new[g] - pi[g]) for g in pi)
        pi = new
        if delta < tol:
            break
    return {g: math.log(v) for g, v in pi.items()}


def to_elo(ratings, base=1500, scale=400.0):
    """対数強さを Elo 風の目盛りに直す。400 点差 = 10 倍強い。"""
    return {g: base + scale * (r / math.log(10)) for g, r in ratings.items()}


In [ ]:
# ⑨ の結果があればそれを、無ければ ⑥ の結果を使う
src = None
if win and win.get("matches"):
    src, src_name = win["matches"], "⑨ 段階的絞り込み"
elif run and run.get("matches"):
    src, src_name = run["matches"], "⑥ 総当たり平均"

bt = elo = {}
if src:
    print(f"{src_name} の {len(src)} 出題から推定します（追加の推論なし）")
    bt = bradley_terry(src)
    elo = to_elo(bt)
    print(f"  レーティングが付いた遺伝子: {len(bt)}")
else:
    print("先に ⑥ か ⑨ を実行してください。")


In [ ]:
btrows = []
if bt:
    seen = Counter()
    for grp, _ in src:
        for g in grp:
            seen[g] += 1
    mean_of = {}
    if win and win.get("matches") is src:
        mean_of = {g: (win["record"].get(g) or {}).get("mean") for g in bt}
    elif run and run.get("matches") is src:
        mean_of = {g: (statistics.mean(run["collected"][g])
                       if run["collected"].get(g) else None) for g in bt}
    order_mean = sorted([g for g in bt if mean_of.get(g) is not None],
                        key=lambda g: -mean_of[g])
    rank_mean = {g: i + 1 for i, g in enumerate(order_mean)}
    for i, g in enumerate(sorted(bt, key=lambda x: -bt[x]), 1):
        btrows.append({
            "rank": i, "gene": g,
            "elo": round(elo[g], 1),
            "n_seen": seen[g],
            "mean_rank": rank_mean.get(g),
            "move": (rank_mean[g] - i) if g in rank_mean else None,
        })

if btrows and HAVE_PANDAS:
    bdf = pd.DataFrame(btrows)[["rank", "gene", "elo", "n_seen", "mean_rank", "move"]]
    try:
        display(bdf.head(30))
    except NameError:
        print(bdf.head(30).to_string(index=False))
elif btrows:
    for r in btrows[:30]:
        print(f"{r['rank']:>5}  {r['gene']:<12}Elo={r['elo']:>7}  "
              f"平均順位={r['mean_rank']}  移動={r['move']}")

if btrows:
    ties = Counter(round(bt[g], 6) for g in bt)
    print(f"\n同点の遺伝子: BT {sum(c for c in ties.values() if c > 1)} 個")
    moved = [r for r in btrows if r["move"] is not None]
    if moved:
        big = sorted(moved, key=lambda r: -abs(r["move"]))[:5]
        print("平均順位から最も動いた 5 個:",
              [(r["gene"], f"{r['mean_rank']}→{r['rank']}") for r in big])


## 数字を信じる前に

- **`n_seen` が小さい行は読まないでください。** 登場回数が数回の遺伝子の平均は、
  どの 25 個と同じ群に入ったかでいくらでも動きます。上位を語るなら、
  少なくとも全遺伝子が 10 回以上登場する回数までまわしてください。
- **これは相対評価です。** スコアは群の中で正規化した確率なので、
  弱い候補ばかりの群に入れば弱い遺伝子でも 1 位になります。絶対的な確信度ではありません。
- **対照を取らないと意味は分かりません。** 疾患名を無関係なものに差し替えて同じ順位が出るなら、
  疾患ではなく知名度を読んでいます。文献の多い遺伝子（TP53, EGFR, TNF）が
  どの疾患でも上位に来ていないかは必ず確認してください。
- **`genelist01_500.txt` は HGNC からの無作為抽出で、特定の疾患用に選んでいません。**
  記号はすべて実在しますが、既知の正解が入っている保証はありません。上位に出た遺伝子は
  「この 500 個の中では相対的に高い」という以上の意味を持ちません。
  実際に使うときは候補リストを差し替えてください。
- **⑤ の動作確認を毎回見てください。** 生成トークンがラベルでないとき、この方式は
  エラーを出さずに沈黙して壊れます。モデルを変えたら必ず確認し直してください。
